In [ ]:
from hashlib import file_digest
from pathlib import Path

import lightning as L
import torch
from IPython.display import Video, display

from training.evaluate import load_checkpoint, record_video
from training.ppo import PPO, PPOData
from training.sim import SKILLS, DuckEnv, hardware

checkpoint_path = Path("m05_factory_walk.pt")
with checkpoint_path.open("rb") as f:
    assert file_digest(f, "sha256").hexdigest() == "f33c6f35659b40d7c7d970bb86ad81db96f5f4de6d55573c832746afba85a91a"


In [ ]:
course_walk = SKILLS["course-walk"]
skills = {"full_reward": course_walk, "no_airtime": course_walk.with_reward_weights(air_time=0.0)}
walk = SKILLS["walk"]
hw = hardware.load("2026-11-26-hd1910")
def start_training(skill):
    L.seed_everything(1)
    net, checkpoint = load_checkpoint(checkpoint_path, "cuda")
    env = DuckEnv(skill, hw, num_envs=2048, device="cuda", seed=1)
    env.set_curriculum_step(checkpoint["common_step_counter"])
    env.reset()
    env.randomize_episode_starts()
    return env, net


In [ ]:
nets, clocks = {}, {}
for label, skill in skills.items():
    env, net = start_training(skill)
    try:
        rewards = env.unwrapped.reward_manager
        print(label, {name: rewards.get_term_cfg(name).weight
                      for name in ("air_time", "track_linear_velocity", "upright")})
        data = PPOData(env, net)
        model = PPO(net)
        trainer = L.Trainer(max_epochs=300, reload_dataloaders_every_n_epochs=1,
                            accelerator="gpu", devices=1, logger=False, enable_checkpointing=False)
        trainer.fit(model, data)
        nets[label] = net
        clocks[label] = env.unwrapped.common_step_counter
    finally:
        env.close()

assert set(clocks.values()) == {25200}
clock = clocks["full_reward"]

In [ ]:
@torch.inference_mode()
def straight_walk(net):
    net = net.eval().to("cuda")
    torch.manual_seed(42)
    env = DuckEnv(walk, hw, num_envs=512, device="cuda", seed=42)
    try:
        env.set_curriculum_step(clock)
        env.reset()
        events = env.unwrapped.event_manager
        if "push_robot" in events.active_terms.get("interval", ()):
            events.get_term_cfg("push_robot").params["velocity_range"] = {"x": (0.0, 0.0), "y": (0.0, 0.0)}
        env.pin_commands(0.30)
        obs, _, _, done, info = env.step(torch.zeros(env.num_envs, env.action_dim, device="cuda"))
        robot = env.unwrapped.scene["robot"]
        active, falls = ~done, info["terminated"].clone()
        for tick in range(2, env.max_episode_length + 1):
            obs, _, _, done, info = env.step(net.act_inference(obs))
            env.pin_commands(0.30)
            falls |= active & info["terminated"]
            active &= ~done
            if tick == 500:
                origin = robot.data.root_link_pos_w[:, :2].clone()
                w, x, y, z = robot.data.root_link_quat_w.unbind(-1)
                yaw = torch.atan2(2 * (w*z + x*y), 1 - 2 * (y*y + z*z))
                heading = torch.stack((yaw.cos(), yaw.sin()), dim=-1)
            if tick == 999:
                forward = ((robot.data.root_link_pos_w[:, :2] - origin) * heading).sum(dim=-1)[active]
        return {"falls": (int(falls.sum()), env.num_envs), "windows_completed": forward.numel(),
                "net_forward_m_median": forward.quantile(0.5).item() if forward.numel() else None}
    finally:
        env.close()

{label: straight_walk(net) for label, net in nets.items()}

In [ ]:
for label, net in nets.items():
    path = Path(f"m05_{label}.mp4")
    record_video(net, walk, hw, path, common_step_counter=clock)
    display(Video(str(path), embed=True))

In [ ]:
for label, net in nets.items():
    torch.save(net.state_dict(), f"m05_{label}.pt")